In [1]:
import matplotlib.pyplot as plt
import geopandas as gpd
import rasterio
from shapely.geometry import Point
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

In [2]:

landover = r"C:\Users\melom\OneDrive\Desktop\Data mining\Cleaned_dataset\Landcover\cleaned_landcover_data.gpkg"
landcover = gpd.read_file(landover)
merged_landcover = landcover[['GRIDCODE', 'geometry']]
print(merged_landcover.head())
print(merged_landcover.info())
print(merged_landcover.crs)

   GRIDCODE                                           geometry
0       210  POLYGON ((6.41528 37.08696, 6.43103 37.0855, 6...
1       210  POLYGON ((7.18084 37.07917, 7.17998 37.08091, ...
2       210  POLYGON ((7.37137 37.08194, 7.3709 37.08717, 7...
3        50  POLYGON ((6.12361 36.68472, 6.12361 36.69306, ...
4       210  POLYGON ((6.26181 37.02361, 6.26193 37.02514, ...
<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 438513 entries, 0 to 438512
Data columns (total 2 columns):
 #   Column    Non-Null Count   Dtype   
---  ------    --------------   -----   
 0   GRIDCODE  438513 non-null  int32   
 1   geometry  438513 non-null  geometry
dtypes: geometry(1), int32(1)
memory usage: 5.0 MB
None
EPSG:4326


# Extraction of the longiture and latitude values - VERSION1

In [3]:
grid_csv = r"C:\Users\melom\OneDrive\Desktop\Data mining\Cleaned_dataset\grid\algeria_tunisia_grid.csv"
grid_df = pd.read_csv(grid_csv)

 
grid_gdf = gpd.GeoDataFrame(
    grid_df,
    geometry=gpd.points_from_xy(grid_df.longitude, grid_df.latitude),
    crs="EPSG:4326"
)

In [4]:
landcover_gdf = merged_landcover

if landcover_gdf.crs != grid_gdf.crs:
    grid_gdf = grid_gdf.to_crs(landcover_gdf.crs)

# Spatial join: assign GRIDCODE from polygon to each point
grid_with_code = gpd.sjoin(
    grid_gdf, 
    landcover_gdf[['GRIDCODE', 'geometry']], 
    how='left', 
    predicate='within')

 
grid_with_code = grid_with_code[['longitude', 'latitude', 'GRIDCODE']]


print(grid_with_code.head())
print(grid_with_code['GRIDCODE'].isna().sum(), "points have no GRIDCODE (outside polygons)")

print(grid_with_code.shape)

grid_with_code.to_csv(r"C:\Users\melom\OneDrive\Desktop\Data mining\Cleaned_dataset\grid\landcover_grid.csv", index=False)


   longitude   latitude  GRIDCODE
0  -1.653868  34.000231     201.0
1  -1.633868  34.000231     201.0
2  -1.613868  34.000231     201.0
3  -1.593868  34.000231     201.0
4  -1.573868  34.000231     151.0
40 points have no GRIDCODE (outside polygons)
(82694, 3)
